In [ ]:
%pip install python-dotenv requests




[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
ECAA***


In [3]:
import os
from dotenv import load_dotenv

load_dotenv()
KEY: str = os.environ["KOREAN_DICT_KEY"]
print(KEY[:4] + "***")

ECAA***


In [4]:
#Q1(a)
import requests
import json
def search_word(q: str, num: int = 10, start: int = 1) -> dict:
    params = {
    "key": KEY, "q": q, "req_type": "json",
    "num": num, "start": start, "type1": "word",
    }
    r = requests.get("https://opendict.korean.go.kr/api/search",
    params=params, timeout=10)
    r.raise_for_status()
    return r.json() 

In [5]:
#(b)
data = search_word("김치")
print(json.dumps(data, ensure_ascii=False, indent=2))

{
  "channel": {
    "total": 328,
    "num": 10,
    "title": "우리말샘 개발 지원(Open API) - 사전 어휘 검색",
    "start": 1,
    "description": "우리말샘 개발 지원(Open API) - 사전 어휘 검색 결과",
    "link": "https://opendict.korean.go.kr",
    "item": [
      {
        "word": "김치",
        "sense": [
          {
            "syntacticArgument": "",
            "syntacticAnnotation": "",
            "cat": "",
            "definition": "소금에 절인 배추나 무 따위를 고춧가루, 파, 마늘 따위의 양념에 버무린 뒤 발효를 시킨 음식. 재료와 조리 방법에 따라 많은 종류가 있다.",
            "link": "https://opendict.korean.go.kr/dictionary/view?sense_no=107717",
            "origin": "",
            "sense_no": "001",
            "target_code": "107717",
            "type": "",
            "pos": "명사"
          }
        ]
      },
      {
        "word": "김-치",
        "sense": [
          {
            "syntacticArgument": "",
            "syntacticAnnotation": "",
            "cat": "인명",
            "definition": "고려 말기·조선 초기의 문신(?~?). 자는 기보(基甫). 김해 부사를 그만두고, 후진 양성에 전

In [6]:
#(c)
items = data["channel"]["item"]
total = data["channel"]["total"]

n = len(items)
print(f"총 {total}건, 이 페이지 {n}건")

for it in items[:5]:
    word = it["word"]
    pos = it["sense"][0].get("pos") if it["sense"][0].get("pos") else "품사 없음"
    long_definition = it["sense"][0]["definition"]
    definition = f"{long_definition[:40]}..." if len(long_definition) > 40 else long_definition
    print(f"{word} ({pos}) --> {definition}")

총 328건, 이 페이지 10건
김치 (명사) --> 소금에 절인 배추나 무 따위를 고춧가루, 파, 마늘 따위의 양념에 버무린...
김-치 (명사) --> 고려 말기·조선 초기의 문신(?~?). 자는 기보(基甫). 김해 부사를 ...
김-치 (명사) --> 조선 중기의 문신(1577~1625). 자는 사정(士精). 호는 남봉(南...
김치 공장 (품사 없음) --> 김치를 만드는 공장.
김치 보릿고개 (품사 없음) --> 김장철인 가을·겨울과 달리 상대적으로 김치가 부족한 봄여름을 비유적으로 ...


설명:
(b)에서 ensure_ascii=False를 추가하지 않으면 기본값은 True라서 한글이 ”\uc5b8\uc5b4”처럼 이스케이프되기 때문에 원문 그대로 하려면 False로 해야한다.
(c)에서는 pos가 sense안에 들어있는 것을 이용하였고 pos가 ""일 수도 있는 것을 감안하여 ""이 none으로 처리되는 것을 이용해 ""이면 품사없음을 if else구문으로 나타내었다. 또한 definition이 40자로 뚝 끊기면 이상해보여서 40자가 넘는 것은 뒤에 "..."까지 붙여봤다.

In [10]:
#Q2(a)
import time
words: list[str] = [
"김치", "라면", "만두", "김밥",
"국수", "떡볶이", "불고기", "비빔밥",
]

word_dict = {}
for q in words:
    data = search_word(q)
    word_dict[q] = data
    total = data["channel"]["total"]
    print(f"{q}: {total}건")
    time.sleep(0.3)

김치: 328건
라면: 86건
만두: 89건
김밥: 39건
국수: 227건
떡볶이: 24건
불고기: 38건
비빔밥: 38건


In [14]:
#(b)
from collections import Counter

l = []
for q in words:
    l.extend(word_dict[q]["channel"]["item"])

pos_list = []
for m in l:
    pos = m["sense"][0].get("pos") or "(미상)"
    pos_list.append(pos)

top3 = Counter(pos_list).most_common(3)

for pos, count in top3:
    print(f"{pos}: {count}회")

명사: 60회
(미상): 19회
어미: 1회


명사가 가장 많은데 우리가 찾은 단어들이 모두 음식(명사)이므로 그와 관련된 단어들도 거의 명사일 확률이 높다.
(a)에서 word_dict를 사용하여 다시 (b)에서 for구문을 다시 사용할 필요없게 했고 counter(pos_list).most_common(3)을 사용하여 가장 count가 많은 3개를 쉽게 고를 수 있다.